In [ ]:
#@title ⚙️ Sel 1: Setup Bersih, Reset Path & Clone Repositori (Final Fix) { display-mode: "form" }
import os
import subprocess

print("🔄 Mengembalikan posisi terminal ke direktori root (/content)...")
# Memaksa Python keluar dari folder yang mungkin sudah terhapus
os.chdir("/content")

print("🧹 Membersihkan folder lama...")
# Menghapus paksa folder Mapperatorinator jika masih tersisa
if os.path.exists("/content/Mapperatorinator"):
    !rm -rf /content/Mapperatorinator

print("📥 Mengunduh repositori Mapperatorinator dari GitHub...")
subprocess.run(["git", "clone", "https://github.com/OliBomby/Mapperatorinator.git"], check=True)

# Berpindah ke folder utama yang baru diunduh
os.chdir("/content/Mapperatorinator")
print("📁 Posisi saat ini:", os.getcwd())

print("📦 Menginstal library bawaan...")
subprocess.run(["pip", "install", "-r", "requirements.txt"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
subprocess.run(["pip", "install", "-e", "."], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

print("📦 Menginstal library khusus Training LoRA...")
subprocess.run(["pip", "install", "-U", "peft", "accelerate", "bitsandbytes", "transformers", "datasets"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

print("✅ Sel 1 Selesai! Mesin bersih dan siap sepenuhnya.")

In [ ]:
#@title 📂 Sel 2: Pengecek & Pembuat Struktur Dataset Otomatis { display-mode: "form" }
import os
import shutil
import json
from google.colab import drive

print("🔗 Menghubungkan ke Google Drive...")
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

dataset_dir = '/content/drive/MyDrive/Mapperatorinator_Training/dataset_osu'

if not os.path.exists(dataset_dir):
    print(f"⚠️ Folder belum ada. Membuat folder baru di: {dataset_dir}")
    os.makedirs(dataset_dir, exist_ok=True)
    print("❌ Folder baru saja dibuat dan masih kosong! Silakan unggah file .osu ke folder tersebut di Google Drive kamu.")
else:
    print(f"📁 Memeriksa isi folder: {dataset_dir}")
    all_files = os.listdir(dataset_dir)

    # Memeriksa file .osu mentah
    file_osu = [f for f in all_files if f.endswith('.osu')]
    # Memeriksa apakah sudah ada folder Track
    folder_track = [f for f in all_files if f.startswith('Track')]

    if len(file_osu) > 0:
        print(f"✅ Ditemukan {len(file_osu)} file .osu mentah. Mengonversinya ke struktur folder Track...")
        for i, nama_file in enumerate(file_osu, start=1):
            track_name = f"Track{i:05d}"
            track_folder = os.path.join(dataset_dir, track_name)
            os.makedirs(track_folder, exist_ok=True)

            # Pindahkan file .osu ke dalam folder track
            sumber = os.path.join(dataset_dir, nama_file)
            tujuan = os.path.join(track_folder, "beatmap.osu")
            if not os.path.exists(tujuan):
                shutil.move(sumber, tujuan)

        # Perbarui ulang daftar folder track setelah dipindah
        folder_track = [f for f in os.listdir(dataset_dir) if f.startswith('Track')]

    if len(folder_track) > 0:
        print(f"🎉 Keren! Ditemukan {len(folder_track)} folder Track siap pakai.")
        print("🛠️ Memperbarui file metadata.json di setiap Track...")

        for folder in folder_track:
            track_path = os.path.join(dataset_dir, folder)
            metadata_file = os.path.join(track_path, "metadata.json")

            # Format dictionary presisi sesuai kebutuhan ors_dataset.py
            metadata_content = {
                "Beatmaps": {
                    "beatmap.osu": {
                        "StandardStarRating": {
                            "0": 3.5
                        }
                    }
                }
            }
            with open(metadata_file, "w") as f:
                json.dump(metadata_content, f, indent=4)

        print("✅ Semua struktur dataset dan metadata.json BERHASIL disiapkan!")
    else:
        print("❌ PERHATIAN: Masih BELUM ADA file .osu atau folder Track sama sekali di dalam folder Google Drive tersebut!")
        print("💡 Solusi: Unggah minimal 1 hingga 10 file .osu ke folder 'dataset_osu' di Google Drive, lalu jalankan ulang sel ini.")

In [ ]:
#@title 🎵 Sel 2 (Fix): Generator File Audio .wav Valid untuk Dataset Training { display-mode: "form" }
import os
import wave
import struct

dataset_dir = '/content/drive/MyDrive/Mapperatorinator_Training/dataset_osu'

print("🔍 Memindai folder Track untuk membuat file audio.wav yang valid...")
if os.path.exists(dataset_dir):
    folders_track = [f for f in os.listdir(dataset_dir) if f.startswith('Track')]

    count = 0
    for folder in folders_track:
        track_path = os.path.join(dataset_dir, folder)
        audio_file_path = os.path.join(track_path, "audio.wav")

        # Membuat file audio .wav standar berdurasi 1 detik yang valid secara matematis
        sample_rate = 22050  # Sample rate standar yang biasa digunakan
        duration = 2.0       # Durasi 2 detik gelombang sunyi/nada

        with wave.open(audio_file_path, 'w') as wav_file:
            wav_file.setnchannels(1)      # Mono
            wav_file.setsampwidth(2)      # 16-bit
            wav_file.setframerate(sample_rate)

            # Menulis data audio kosong (silence) yang valid
            num_samples = int(sample_rate * duration)
            for _ in range(num_samples):
                wav_file.writeframes(struct.pack('<h', 0))

        # Hapus file audio.mp3 palsu sebelumnya jika ada agar tidak konflik
        mp3_legacy = os.path.join(track_path, "audio.mp3")
        if os.path.exists(mp3_legacy):
            os.remove(mp3_legacy)

        count += 1

    print(f"✅ Berhasil membuat file 'audio.wav' yang valid di dalam {count} folder Track!")
else:
    print("❌ Folder dataset tidak ditemukan di Google Drive!")

In [ ]:
#@title 🚀 Sel 3: Training LoRA MonstrataTest (Bypass Mutlak Wandb) { display-mode: "form" }

#@markdown #### Pengaturan Identitas LoRA
nama_mapper = "MonstrataTest" # @param {type:"string"}
#@markdown #### Pengaturan Training Uji Coba
epochs = 2 # @param {type:"integer"}
batch_size = 2 # @param {type:"integer"}

import os
import subprocess
import yaml
import json

repo_dir = "/content/Mapperatorinator"
os.chdir(repo_dir)

dataset_dir = '/content/drive/MyDrive/Mapperatorinator_Training/dataset_osu'
output_dir = f'/content/drive/MyDrive/Mapperatorinator_Training/lora_output/{nama_mapper}'
os.makedirs(output_dir, exist_ok=True)

# 1. Pindai folder Track dan perbarui metadata.json
print("🛠️ Memastikan struktur metadata.json di dalam 10 folder Track sudah valid...")
folders_track = [f for f in os.listdir(dataset_dir) if f.startswith('Track')]
jumlah_file = len(folders_track)

for folder in folders_track:
    track_path = os.path.join(dataset_dir, folder)
    metadata_file = os.path.join(track_path, "metadata.json")

    metadata_content = {
        "Beatmaps": {
            "beatmap.osu": {
                "StandardStarRating": {
                    "0": 3.5
                }
            }
        }
    }
    with open(metadata_file, "w") as f:
        json.dump(metadata_content, f, indent=4)

if jumlah_file == 0:
    print("❌ ERROR: Tidak ada folder Track ditemukan!")
else:
    print(f"✅ Ditemukan {jumlah_file} folder Track siap training.")
    print("🔧 Menyesuaikan konfigurasi v30.yaml...")

    config_path = "configs/train/v30.yaml"
    with open(config_path, "r") as f:
        config_data = yaml.safe_load(f)

    # Menyesuaikan path & jangkauan dataset
    config_data['data']['train_dataset_path'] = dataset_dir
    config_data['data']['test_dataset_path'] = dataset_dir
    config_data['data']['train_dataset_start'] = 0
    config_data['data']['train_dataset_end'] = jumlah_file
    config_data['data']['test_dataset_start'] = 0
    config_data['data']['test_dataset_end'] = jumlah_file

    # Optimasi batch size & grad_acc
    config_data['optim']['batch_size'] = batch_size
    current_grad_acc = config_data['optim'].get('grad_acc', 4)
    if batch_size < current_grad_acc or batch_size % current_grad_acc != 0:
        config_data['optim']['grad_acc'] = 1

    # Jalur mappers_path absolut
    json_absolute_path = os.path.join(repo_dir, "datasets", "beatmap_users.json")
    os.makedirs(os.path.dirname(json_absolute_path), exist_ok=True)
    if not os.path.exists(json_absolute_path):
        with open(json_absolute_path, "w") as f:
            json.dump({}, f)
    config_data['data']['mappers_path'] = json_absolute_path

    with open(config_path, "w") as f:
        yaml.dump(config_data, f)

    # 2. Bedah dan matikan total baris Wandb di train.py secara otomatis
    print("🛠️ Menyunting file osuT5/train.py untuk mencopot modul Wandb...")
    train_py_path = "osuT5/train.py"
    with open(train_py_path, "r") as f:
        lines = f.readlines()

    new_lines = []
    skip_lines = False
    for line in lines:
        if "accelerator.init_trackers(" in line:
            skip_lines = True
            new_lines.append("    # [DIHAPUS OTOMATIS] " + line)
            continue
        if skip_lines:
            new_lines.append("    # [DIHAPUS OTOMATIS] " + line)
            if ")" in line:
                skip_lines = False
            continue
        new_lines.append(line)

    with open(train_py_path, "w") as f:
        f.writelines(new_lines)
    print("✅ Modul Wandb berhasil dicopot dari kode sumber!")

    print(f"🔥 Memulai proses training LoRA untuk {nama_mapper}...")
    print("📊 Menampilkan Live Progress Bar (Tunggu sebentar hingga loading bar muncul)...\n")
    print("-" * 50)

    try:
        env = os.environ.copy()
        env["WANDB_DISABLED"] = "true"

        training_command = [
            "python", "osuT5/train.py",
            "-cn", "v30"
        ]

        process = subprocess.Popen(
            training_command,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            env=env,
            bufsize=1
        )

        for line in process.stdout:
            print(line, end="")

        process.wait()

        print("-" * 50)

        if process.returncode == 0:
            print(f"\n🎉 TRAINING SELESAI DENGAN SUKSES!")
            print(f"Hasil LoRA kamu tersimpan di: {output_dir}")
        else:
            print(f"\n❌ Terjadi kesalahan saat proses training (Exit Code: {process.returncode})")

    except Exception as e:
        print(f"❌ Error sistem: {e}")

In [ ]:
#@title 🔄 Sel 2 (Final & Pasti Sinkron): Penyelaras Metadata Otomatis { display-mode: "form" }
import os
import json

dataset_dir = '/content/drive/MyDrive/Mapperatorinator_Training/dataset_osu'

print("🔍 Mensinkronkan nama file .osu asli dengan metadata.json...")
if os.path.exists(dataset_dir):
    folders_track = [f for f in os.listdir(dataset_dir) if f.startswith('Track')]

    count = 0
    for folder in folders_track:
        track_path = os.path.join(dataset_dir, folder)
        beatmaps_dir = os.path.join(track_path, "beatmaps")

        if os.path.exists(beatmaps_dir):
            osu_files = [f for f in os.listdir(beatmaps_dir) if f.endswith('.osu')]

            if len(osu_files) > 0:
                real_file_name = osu_files[0] # Ambil nama file .osu apa adanya

                metadata_file = os.path.join(track_path, "metadata.json")

                # Gunakan nama file asli sebagai key di dalam dictionary agar tidak KeyError
                metadata_content = {
                    "Beatmaps": {
                        real_file_name: {
                            "Index": 0,
                            "StandardStarRating": {
                                "0": 3.5
                            }
                        }
                    }
                }

                with open(metadata_file, "w") as f:
                    json.dump(metadata_content, f, indent=4)
                count += 1

    print(f"✅ Berhasil menyelaraskan metadata untuk {count} folder Track!")
else:
    print("❌ Folder dataset tidak ditemukan di Google Drive!")